In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",api_key=groq_api_key)

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
    ]
)

AIMessage(content='Hello Ishaan! 👋 It’s great to meet a chief AI Engineer. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Ishaan and I am a chief Ai Engineer". Likely they want a response. There\'s no request for disallowed content. We just greet and respond. Possibly ask how can I help.'}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 85, 'total_tokens': 165, 'completion_time': 0.169162342, 'completion_tokens_details': {'reasoning_tokens': 48}, 'prompt_time': 0.031660661, 'prompt_tokens_details': None, 'queue_time': 0.398221443, 'total_time': 0.200823003}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f7ba-db65-7022-88fd-cab30c5c880d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 80, 'total_tokens': 165, 'output

In [7]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
        AIMessage(content="Hi Ishaan, nice to meet you! How can I assist you today?"),
        HumanMessage(content="What is my name and what do I do?")
    ]
)   

AIMessage(content='Your name is **Ishaan**, and you work as a **Chief AI Engineer**.', additional_kwargs={'reasoning_content': 'The user asks: "What is my name and what do I do?" They previously introduced themselves: "Hi, my name is Ishaan and I am a chief Ai Engineer". So answer: name is Ishaan, role is chief AI Engineer. Should respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 121, 'total_tokens': 204, 'completion_time': 0.174350066, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.029930714, 'prompt_tokens_details': None, 'queue_time': 0.465593037, 'total_time': 0.20428078}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f7ba-e27e-7b63-9591-bbebaa5e2874-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 83, 'total_tokens': 

###  Message History
We can use a Message history class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use it

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


with_message_history=RunnableWithMessageHistory(
    model,
    get_session_history
)


/home/ishaan-yadav/Langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [2]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
    ],
    config=config
)

In [11]:
with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config
)

AIMessage(content='Your name is **Ishaan**.', additional_kwargs={'reasoning_content': 'User asks "What is my Name?" We have prior conversation: user introduced themselves as "Ishaan". So answer: your name is Ishaan.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 218, 'total_tokens': 268, 'completion_time': 0.106248365, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.054265473, 'prompt_tokens_details': None, 'queue_time': 7.165300018, 'total_time': 0.160513838}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f7bd-279a-7cf0-b0f2-35fdecc9a8fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 218, 'output_tokens': 50, 'total_tokens': 268, 'output_token_details': {'reasoning': 32}})

In [14]:
## change the session id to start a new conversation
config1={"configurable":{"session_id":"chat2"}}

response=with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config1
)
response.content

'I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [15]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hey my name is John"),
    ],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [16]:
response=with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config1
)
response.content

'Your name is John.'